# 06-NE555 定时器与振荡电路


## 1. 本篇学习范围

本篇对应课程视频 **40—51**：NE555 简介与引脚、比较器、反相器、或非门、双稳态触发器、内部原理图、外围电路、方波产生、电子琴原理，以及模拟器中的 NE555。

这一篇把前面学过的分压、电容充放电和三极管开关连成一个完整系统：

```text
内部电阻分压 → 提供两个参考电压
                      ↓
电容电压变化 → 比较器判断是否越过阈值
                      ↓
触发器记住状态 → 控制输出和放电开关
                      ↓
电容反复充放电 → 输出连续方波 → 改变频率可以改变音高
```

| 课程内容 | 要解决的问题 |
|---|---|
| NE555 与引脚 | 每个引脚负责什么，怎样看实物脚号？ |
| 比较器与逻辑门 | 怎样把电压大小关系变成高、低状态？ |
| 双稳态触发器 | 为什么输入不再变化后，输出还能保持原状态？ |
| 外围 RC 电路 | 电容从哪里充电，又通过哪里放电？ |
| 方波与频率 | 周期、频率、占空比怎样计算？ |
| 电子琴 | 按不同按键，为什么会发出不同音高？ |

> 本篇以课程目录和配套讲义为主。供电与引脚接口说明、公式推导、计算练习、单稳态简介和实际负载驱动属于学习补充。讲义中部分阈值翻转方向和 RC 时间的表述，在对应位置作了澄清。


## 2. NE555 是什么：把比较、记忆和开关集成在一起

NE555 是一种常见的定时器集成电路。外接电阻、电容后，可以实现脉冲、延时或连续振荡。

| 常见用途 | 输出行为 |
|---|---|
| 单次定时 | 一次触发后输出持续一段时间的脉冲 |
| 连续振荡 | 周期性切换高、低电平 |
| 频率可调信号 | 改变 RC 参数，改变输出频率 |
| 简单发声 | 用音频范围的周期信号驱动适当的发声电路 |

理解 NE555，不必先记住全部内部晶体管，只需先认识五个功能部分：电阻分压、两个比较器、状态记忆、放电开关、输出驱动。

### 供电和输出先看具体型号

本篇使用 **5 V** 作为入门电路的练习供电。TI 的 NE555 产品说明给出了 `5～15 V` 的工作说明；具体器件的推荐工作范围应以完整型号的数据手册为准。

- 普通双极型 NE555 与 CMOS 555 的供电范围、耗电和输出能力不同。
- 不能看到“555”就认定它可在 `3.3 V` 下可靠工作。
- 输出高电平不一定等于 VCC，输出低电平也不是完全理想的 0 V；它们与负载和器件有关。

讲义强调 NE555 是很好的综合学习案例。掌握它能帮助理解模拟与数字功能怎样协同工作，后续仍需要继续学习其他电路和器件。


## 3. 八个引脚：先查脚号，再按功能接线

| 脚号 | 常见名称 | 中文 | 在本篇振荡电路中的作用 |
|---|---|---|---|
| 1 | GND | 地 | 接供电负端、作为电压参考 |
| 2 | TRIG | 触发输入 | 电压低于下阈值时，促使输出变高 |
| 3 | OUT | 输出 | 输出高、低电平交替的波形 |
| 4 | RESET | 复位输入 | 低有效；正常振荡时接到适当高电平，常接 VCC |
| 5 | CONT / CTRL | 控制电压 | 改变内部参考电压；不使用时常按推荐接电容到地 |
| 6 | THRES / THRESH | 阈值输入 | 电压超过上阈值时，促使输出变低 |
| 7 | DISCH | 放电端 | 内部开关导通时向地提供放电通路 |
| 8 | VCC | 正电源 | 接合适的直流正电源 |

### 常见 DIP-8 顶视图

```text
          缺口朝上
        ┌───∪───┐
   GND 1│       │8 VCC
  TRIG 2│ NE555 │7 DISCH
   OUT 3│       │6 THRES
 RESET 4│       │5 CONT
        └───────┘
```

这是常见封装的顶视图示意，实物方向与封装仍需查手册。顶视图和底视图不能混用；原理图符号上的引脚摆放，也不必与实物左右位置相同。

**两个容易混淆的点：**2 脚和 6 脚都是比较器输入；7 脚是放电开关端，不是第二个方波输出。


## 4. 比较器：判断两个输入电压谁更大

比较器有同相输入 `+` 和反相输入 `-`。这里的加减号表示比较关系，不是电源正负极。

在一个简化模型中：

| 输入关系 | 比较器输出 |
|---|---|
| V+ > V- | 趋向高状态 |
| V+ < V- | 趋向低状态 |
| 两者非常接近 | 实际行为受失调、噪声和器件特性影响 |

例如参考电压为 `3 V`，被测电压接 `+`、参考接 `-`：被测电压超过参考时输出变高；交换两路输入后，判断方向也会反过来。

### 它在 NE555 中负责什么

- 上方比较器检查 THRES 是否超过约 `2/3 VCC`。
- 下方比较器检查 TRIG 是否低于约 `1/3 VCC`。
- 比较器的结果进入状态记忆电路，再控制 OUT 和放电端。

**必要区分：**课程把这一节称为“比较器（运放）”，用于介绍差分输入的比较关系。运算放大器常在负反馈下做线性放大，专用比较器用于状态判断，二者不能在所有电路中直接互换；NE555 内部这里应按比较器理解。


## 5. 反相器与或非门：理解内部逻辑

逻辑 `0/1` 对应器件允许范围内的低、高状态，并不总是固定的 `0 V/5 V`。

### 反相器，也叫非门

```text
Y = NOT A
```

| A | Y |
|---|---|
| 0 | 1 |
| 1 | 0 |

课程视频文件名为“相反器”，这里统一使用常见名称“反相器”。符号中的小圆圈通常表示取反或低有效。

### 或非门 NOR

先对输入做“或”，再把结果取反：

```text
Y = NOT (A OR B)
```

| A | B | Y |
|---|---|---|
| 0 | 0 | 1 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 0 |

记忆方法：**只有所有输入都是 0，或非门才输出 1。**

单个逻辑门只按当前输入决定输出；把门电路交叉反馈连接起来，才可以形成带记忆的电路。


## 6. 双稳态触发器：让电路记住刚才的状态

课程使用两个或非门交叉反馈来解释 SR 状态记忆。这里按高有效的普通 NOR 型 SR 锁存器理解：

- S：Set，置位，使 Q 成为 1。
- R：Reset，复位，使 Q 成为 0。
- S、R 都回到 0 后，Q 保持之前的状态。

| S | R | Q 的行为 |
|---|---|---|
| 0 | 0 | 保持原状态 |
| 1 | 0 | 置位为 1 |
| 0 | 1 | 复位为 0 |
| 1 | 1 | 普通 NOR 型锁存器的禁用组合，不作为正常工作条件 |

“双稳态”表示电路可以稳定地保持 0 或 1。它不需要输入信号一直维持“置位”或“复位”，这就是记忆功能。

### 为什么 NE555 需要记忆

当电容电压位于上下阈值之间时，不能只问“当前电压是多少”，还要知道电容正在充电还是放电。触发器保存的状态，决定接下来继续哪个过程。

> 这里是理解记忆功能的教学模型，不是时钟触发 D 触发器。普通 NOR 锁存器的禁用组合也不能直接当作 NE555 实物的输入优先级表；实物应按厂商给出的功能说明判断。


## 7. 内部电阻分压：1/3 VCC 与 2/3 VCC 从哪里来

讲义功能框图画出三个相等的电阻，从 VCC 串联到地。经典双极型 555 的教学图常把它们标为 `5 kΩ`。

```text
VCC
 │
 R
 │── 上参考电压：约 2/3 VCC ── CONT（5 脚）
 R
 │── 下参考电压：约 1/3 VCC
 R
 │
GND
```

三个相同电阻平均分担电源电压，因此形成两个比较基准。

当 `VCC=5 V` 时：

```text
下阈值 V_L ≈ 5 / 3 ≈ 1.67 V
上阈值 V_H ≈ 2 × 5 / 3 ≈ 3.33 V
```

在未外部改变 CONT 电压的标准条件下：

- TRIG 低于下阈值，使输出进入高状态。
- THRES 高于上阈值，且触发输入不再处于低触发条件，使输出进入低状态。
- 两个判断都不要求改变状态时，保持之前的状态。

5 脚可以改变参考电压。外部控制电压参与工作后，不能再机械套用默认的 `1/3、2/3 VCC` 和标准定时公式。


## 8. 从输入到输出：把状态与放电开关对应起来

下面依据 TI NE555 的功能说明作简化。暂不考虑阈值附近的误差、延迟和外部控制电压。

| RESET | TRIG | THRES | OUT | 7 脚内部放电开关 |
|---|---|---|---|---|
| 低 | 任意 | 任意 | 低 | 导通 |
| 高 | 低于约 1/3 VCC | 任意 | 高 | 关断 |
| 高 | 高于约 1/3 VCC | 高于约 2/3 VCC | 低 | 导通 |
| 高 | 高于约 1/3 VCC | 低于约 2/3 VCC | 保持之前的状态 | 按之前的状态保持 |

RESET 是强制复位，优先于普通触发与阈值输入。具体型号或仿真模型仍应核对其说明；不要把理想表中的“等于阈值”当成精确、无噪声的稳定边界。

### 在标准振荡电路中

2、6 脚连接同一个电容节点，记它的电压为 `V_C`：

```text
V_C < 下阈值  → OUT 高 → 放电开关关断 → 开始/继续充电
V_C > 上阈值  → OUT 低 → 放电开关导通 → 开始放电
上下阈值之间 → 保持状态，继续原来的充电或放电过程
```

**对讲义的澄清：**第 79 页文字中关于上下阈值对应的 OUT 翻转方向有颠倒。本篇统一按功能框图和厂商说明理解：到达上阈值后高转低，到达下阈值后低转高。


## 9. 标准无稳态电路：外围电阻、电容怎样连接

“无稳态”表示电路不会一直停在高或低状态，而是自动重复切换。课程给出的标准接法是：

```text
VCC ── R_A ──●── R_B ──●── C ── GND
              │         │
           7 DISCH    2 TRIG
                        │
                     6 THRES

8 VCC   → 正电源
1 GND   → 地
4 RESET → VCC（正常振荡）
5 CONT  → 滤波电容 → GND（按推荐应用接法）
3 OUT   → 观察波形或连接合适的负载接口
```

这里：

- `R_A` 位于 VCC 与 7 脚之间。
- `R_B` 位于 7 脚与电容节点之间。
- 2、6 脚短接后连接电容节点，电容另一端接地。
- 7 脚的内部放电开关控制该节点的放电路径。

### 课程第 77 页给出的参数

| 元件 | 课程示例值 |
|---|---|
| R_A | 1 kΩ |
| R_B | 100 kΩ |
| 定时电容 C | 100 nF |
| 5 脚滤波电容 | 0.01 μF，即 10 nF |
| RESET | 拉高，使芯片正常工作 |

实际搭建时，还应在 8、1 脚附近放合适的电源去耦电容，例如常见的 `100 nF` 陶瓷电容。电源去耦、5 脚滤波和定时电容分别承担不同作用，不能把三个电容的接点混为一谈。

`R_A` 不能直接替换成短路；放电开关导通时，VCC 会经 R_A 向 7 脚提供电流，需要限制它。


## 10. 一个振荡周期：先充电，再放电

### 阶段一：输出高，电容充电

7 脚内部放电开关关断，电容通过两个电阻充电：

```text
VCC → R_A → R_B → C → GND
```

电容电压从约 `1/3 VCC` 上升到约 `2/3 VCC`，OUT 保持高。

### 阶段二：到达上阈值，输出转低

THRES 比较器使状态翻转，OUT 变低，7 脚放电开关导通。

### 阶段三：输出低，电容放电

电容主要通过 R_B 和 7 脚开关放电：

```text
C 上端 → R_B → 7 脚内部放电开关 → GND
```

电容电压从约 `2/3 VCC` 下降到约 `1/3 VCC`，OUT 保持低。此时 R_A 也向已被拉低的 7 脚提供电流，但不属于定时电容的主要放电电阻。

### 阶段四：到达下阈值，输出转高

TRIG 比较器使状态再次翻转，放电开关关断，重新开始充电。

| 过程 | OUT | 7 脚开关 | 电容电压 | 定时路径 |
|---|---|---|---|---|
| 充电 | 高 | 关断 | 下阈值 → 上阈值 | R_A + R_B |
| 放电 | 低 | 导通 | 上阈值 → 下阈值 | R_B |

**波形要分开看：**OUT 是高、低交替的近似方波；电容节点是指数充放电曲线，不能把它当成同一条方波或严格的直线三角波。


## 11. 周期、频率与占空比

以下公式适用于上面的标准接法，默认参考电压约为 `1/3、2/3 VCC`，放电开关和元件近似理想。

```text
t_H ≈ 0.693 × (R_A + R_B) × C
t_L ≈ 0.693 × R_B × C

T = t_H + t_L ≈ 0.693 × (R_A + 2R_B) × C
f = 1 / T ≈ 1.443 / [(R_A + 2R_B) × C]

高电平占空比 D_H = t_H / T = (R_A + R_B) / (R_A + 2R_B)
低电平时间比例 D_L = t_L / T = R_B / (R_A + 2R_B)
D_H + D_L = 1
```

| 量 | 含义 | 常用单位 |
|---|---|---|
| t_H | 每周期输出高电平的时间 | s、ms |
| t_L | 每周期输出低电平的时间 | s、ms |
| T | 一个完整充放电周期 | s |
| f | 每秒周期数 | Hz |
| D_H | 高电平占整个周期的比例 | 比值或百分数 |

计算时统一使用 `Ω、F、s`，再换算显示单位：`100 kΩ=100000 Ω`，`100 nF=100×10⁻⁹ F`。

### 占空比必须先说明定义

本篇“占空比”默认指 OUT 的高电平比例。若资料写 `R_B/(R_A+2R_B)`，它描述的是低电平时间比例，不能与高电平占空比混用。

对标准接法且 `R_A>0`，`D_H>50%`；当 R_A 远小于 R_B 时，它可以接近 50%。要改变这一限制，需要改变充放电路径等电路结构，不能直接把 R_A 短路来追求 50%。


## 12. 0.693 从哪里来：区分定时时间与 RC 时间常数

`0.693` 约等于自然对数 `ln(2)`。它在这里出现，是因为电容在两个特定阈值之间变化。

### 充电过程

电容向 VCC 充电时，离最终电压的差值按指数减小。从 `1/3 VCC` 到 `2/3 VCC`：

```text
起点的差值：VCC - 1/3 VCC = 2/3 VCC
终点的差值：VCC - 2/3 VCC = 1/3 VCC
差值缩小到一半 → exp[-t / ((R_A+R_B)C)] = 1/2
所以 t_H = ln(2) × (R_A+R_B)C
```

### 放电过程

电容向地放电，从 `2/3 VCC` 降到 `1/3 VCC`，电压也变为原来的一半：

```text
exp[-t / (R_B C)] = 1/2
所以 t_L = ln(2) × R_B C
```

### 时间常数是 τ=RC

一个普通 RC 电路，从 0 V 向固定电压充电，在 **t=RC** 时达到最终电压的约 `63.2%`；在 `t=0.693RC` 时达到约 `50%`。

因此，讲义第 80 页把“达到 63.2%”与 `0.693RC` 联系起来的表述需要澄清：时间常数是 RC，NE555 的 `0.693RC` 来自其阈值间的指数变化，完整周期则包含充电和放电两段。

上电时若电容从 0 V 开始，第一次充电与稳态的 `1/3→2/3 VCC` 不完全相同；标准周期公式主要描述稳定振荡后的周期。


## 13. 用课程参数算一次：1 kΩ、100 kΩ、100 nF

按第 77 页的外围参数：

```text
R_A = 1 kΩ = 1000 Ω
R_B = 100 kΩ = 100000 Ω
C = 100 nF = 100 × 10⁻⁹ F
```

代入标准公式：

```text
t_H ≈ 0.693 × (1000+100000) × 100×10⁻⁹ ≈ 7.00 ms
t_L ≈ 0.693 × 100000 × 100×10⁻⁹ ≈ 6.93 ms
T ≈ 13.93 ms
f ≈ 1 / 0.01393 ≈ 71.8 Hz

D_H = 101000 / 201000 ≈ 50.25%
D_L = 100000 / 201000 ≈ 49.75%
```

这组数值说明：

- 一个周期约 14 ms，高低持续时间比较接近。
- 电容电压在 5 V 供电下，稳态大致往返于 `1.67 V` 与 `3.33 V`。
- OUT 每秒重复约 72 次，用示波器或仿真波形观察更合适；LED 未明显闪烁不能据此判断没有振荡。

实际频率会受到电阻、电容误差、漏电、阈值误差和放电开关压降等影响；这里的数值是理想计算，不是实测记录。


## 14. 改变一个元件，观察频率怎样变化

下表为同一标准接法的计算练习，供电和阈值条件保持不变。

| R_A | R_B | 定时 C | 预计频率 | 高电平占空比 |
|---|---|---|---|---|
| 1 kΩ | 100 kΩ | 100 nF | 约 71.8 Hz | 约 50.25% |
| 1 kΩ | 100 kΩ | 1 μF | 约 7.18 Hz | 约 50.25% |
| 1 kΩ | 100 kΩ | 10 μF | 约 0.718 Hz | 约 50.25% |
| 1 kΩ | 10 kΩ | 100 nF | 约 687 Hz | 约 52.38% |

### 从公式读出变化方向

- 只增大 C：高、低时间按相同比例增加，频率下降；理想占空比不变。
- 增大 R_B：充电和放电都变慢，频率下降，占空比也随之变化。
- 增大 R_A：主要增加充电时间，使高电平时间比例增大。

用 `10 μF` 做低频 LED 闪烁练习时，一个周期约 `1.39 s`。若使用有极性的电解电容，定时节点接其正端、地接负端，并检查耐压和漏电情况。

> “增大 RC 就变慢”可以用来记趋势，计算时仍需分清充电电阻 R_A+R_B 与放电电阻 R_B。元件极端大或小时，简单模型的误差可能明显增加。


## 15. 三种工作方式：无稳态、单稳态、双稳态

课程重点是连续振荡。为了理解“定时器”的其他用途，这里作必要补充。

| 工作方式 | 行为 | 典型理解 |
|---|---|---|
| 无稳态 | 持续自动在高、低状态间切换 | 连续脉冲、闪烁、发声 |
| 单稳态 | 平时保持一个状态，触发后输出定时脉冲，再返回 | 延时、单次脉冲 |
| 双稳态 | 可以保持高或低，由相应输入改变状态 | 状态记忆 |

### 单稳态的常见定时关系

标准单稳态电路中，定时电容从近似 0 V 经电阻充到约 `2/3 VCC`：

```text
t ≈ ln(3) × R × C ≈ 1.1RC
```

例如 `R=100 kΩ`、`C=10 μF`，理想脉冲宽度约为 `1.1 s`。

其典型接法与本篇无稳态电路不同：6、7 脚连接定时节点，电阻从 VCC 接该节点，电容接地，2 脚另外接合适的短低触发脉冲。

该关系要求触发输入及时返回非触发状态，复位保持无效，并满足具体器件的工作条件。不能把按钮长时间拉低、不同接线或不同控制阈值下的输出都直接认为是 `1.1RC`。


## 16. NE555 电子琴：按键选择电阻，电阻选择频率

讲义第 81、82 页把振荡频率与音高联系起来。其基本过程是：

```text
按下某个按键
      ↓
选中电阻链的一段，使等效 R_B 改变
      ↓
电容充放电周期改变
      ↓
OUT 输出不同频率的方波
      ↓
经过合适的发声驱动电路，产生不同音高
```

R_A 和 C 固定时，R_B 越大，频率越低；R_B 越小，频率越高。

### 怎样读课程电子琴图

第 82 页的图中，R_A 为 `1 kΩ`，定时电容 C2 标记 `104`，即 `100 nF`；按键选择由 `10 kΩ`、`1 kΩ` 和若干 `2 kΩ` 组成的电阻链。

靠近定时节点的按键可选出较短的电阻路径，靠前的按键会把更多串联电阻纳入路径。要先沿实际导线确认选中的总阻值，再代入公式；多个按键同时按下时，等效路径还可能变化。

这组简单阻值用于展示频率变化，不能直接视为每个按键都精准对应标准音阶。标准相邻音的频率比例也不是“每个按键增加相同电阻”就能准确实现。

### 以常见 C 大调的一组音高为参考

以下按 A4 为 440 Hz 的常见定音列出，供理解音高关系；频率相差一倍对应一个八度。

| 简谱级数 | 音名 | 参考频率 |
|---|---|---|
| 1 | C4 | 261.63 Hz |
| 2 | D4 | 293.66 Hz |
| 3 | E4 | 329.63 Hz |
| 4 | F4 | 349.23 Hz |
| 5 | G4 | 392.00 Hz |
| 6 | A4 | 440.00 Hz |
| 7 | B4 | 493.88 Hz |

### 练习：为 440 Hz 反算电阻

另外选择练习参数 `R_A=1 kΩ`、`C=10 nF`，则：

```text
R_B ≈ [1 / (0.693 × f × C) - R_A] / 2
代入 f=440 Hz，R_B 约为 163.5 kΩ
```

取约 `163 kΩ` 时，理想频率约 `441 Hz`。若要调到目标音高，可以结合固定电阻与微调电阻，再根据实测频率调整。

这组反算参数是学习补充，定时 C 与课程电子琴图的 `100 nF` 不同。电容误差、温度漂移等也会影响音准；简单 NE555 电子琴主要用于理解振荡和发声。


## 17. 从方波到声音：检查发声器件和输出负载

电信号频率决定发声电路的振动频率，但音量还与驱动幅度、器件效率和安装方式有关；方波的谐波也会影响音色。

| 发声器件 | 使用时需要关注 |
|---|---|
| 有源蜂鸣器 | 内部已有振荡电路，通常靠通断控制发声；不适合直接当作任意音高转换器 |
| 无源压电发声器 | 可用外部周期信号驱动，核对频率、允许电压和接口要求 |
| 低阻动圈喇叭 | 电流需求可能较大，需要匹配输出能力，通常考虑合适的音频驱动电路 |

讲义电子琴图把 OUT 经 `4.7 μF` 电容连接喇叭。串联电容用于隔离直流分量，但它本身不会提升芯片的输出电流能力。

### 把输出能力与声音效果分开检查

- OUT 带载后仍需满足允许电流、输出压降和芯片功耗条件。
- 不能因为芯片“可以源/灌电流”，就把任何低阻喇叭直接接上。
- 使用有极性的耦合电容时，要结合两端直流偏置核对极性与耐压。
- 若只验证振荡原理，先用波形观察或带限流电阻的 LED，再加入发声接口。

LED 接在 OUT 与地之间时，高电平通常点亮；若接成 VCC → 电阻 → LED → OUT，则低电平时灌电流点亮。读“亮灯时间”时要把 LED 接法与 OUT 高电平占空比分清。


## 18. 在模拟器中复现：同时观察 OUT 和电容电压

课程视频 51 介绍模拟器中的 NE555。下面给出复习用的操作流程，菜单和元件模型名称以所用 CircuitJS 版本为准。

### 优先用 NE555 模型验证外围电路

1. 放置 NE555 模型，按第 9 节接电源、复位和定时 RC。
2. 先使用 `5 V、R_A=1 kΩ、R_B=100 kΩ、C=100 nF`。
3. 给 3 脚 OUT 和 2/6 脚电容节点分别添加波形观察。
4. 等待进入稳定振荡，记录高时间、低时间、完整周期和电容上下限。
5. 一次只改变一个参数，把观察结果与第 13、14 节的计算比较。

| 观察对象 | 预期表现 |
|---|---|
| OUT | 稳态周期约 13.9 ms，高低交替 |
| 电容电压 | 稳态大致在 1.67～3.33 V 之间充放电 |
| 7 脚 | 充电时内部开关关断，放电时被拉到接近地电位 |
| RESET 拉低 | OUT 转低，放电开关导通，振荡停止 |

### 也可以按课程逐模块搭建内部功能模型

用三个等值电阻形成参考，再连接两个比较器、SR 状态记忆、输出逻辑和放电开关：

- THRES 比较器：THRES 接 `+`，上参考接 `-`，越过上阈值时提出复位请求。
- TRIG 比较器：下参考接 `+`，TRIG 接 `-`，低于下阈值时提出置位请求。
- 状态逻辑控制 OUT，同时让低输出状态对应放电开关导通。

普通门电路模型、放电晶体管驱动和真实 NE555 的优先级、输出级并不完全相同；逐模块搭建主要用于理解功能关系，放电开关采用 BJT 时还应按第 05 篇限制基极驱动电流。

### 记录表

| 参数组 | 预计 f | 仿真/实测 t_H | t_L | f | 电容下限/上限 |
|---|---|---|---|---|---|
| 课程参数 | 71.8 Hz | 待记录 | 待记录 | 待记录 | 待记录 |
| 仅将 C 改为 1 μF | 7.18 Hz | 待记录 | 待记录 | 待记录 | 待记录 |
| 仅将 C 改为 10 μF | 0.718 Hz | 待记录 | 待记录 | 待记录 | 待记录 |

> 表中的“预期”来自电路分析；本篇未实际运行并记录这些仿真测量，因此不把预测值写成已经取得的实验结果。


## 19. 实际接线与故障排查

| 现象 | 优先检查 |
|---|---|
| 一直低电平 | RESET 是否被拉低？供电和脚号是否正确？电容节点是否被异常拉高？ |
| 一直高电平 | 定时节点是否能充到上阈值？2/6 脚、R_A、R_B、C 是否连对？ |
| 周期差很多 | 单位是否换算正确？电容容量与极性是否正确？电阻链总值是否算对？ |
| 波形抖动或误触发 | 电源去耦、接地、复位与控制输入状态是否合适？负载是否影响供电？ |
| 加喇叭后异常 | 输出是否过载？耦合电容、驱动电路和电源是否合适？ |
| LED 看起来常亮 | 振荡是否太快？LED 是高电平点亮还是低电平点亮？ |
| 芯片或元件发热 | 是否错脚、短路或超负载？放电端是否缺少必要限流？ |

### 与前几篇知识连起来

- 按第 03 篇方法，先断电检查接线，再用万用表并联测供电与关键节点电压。
- 万用表对动态节点可能显示平均值；它通常不能仅凭一个读数证明输出是正确方波。
- 按第 05 篇检查外部 BJT/MOS 驱动条件；感性负载还需考虑第 04 篇的续流保护。
- NE555 以 5 V 供电时，若 OUT 要接到 3.3 V 单片机输入，应核对两侧逻辑电平与引脚允许电压，必要时使用电平转换。

**对课程图的应用边界：**第 82 页供电连接标有 `3～9 V`，不能据此确认所有普通 NE555 在 3 V 都满足规格；本篇先采用 5 V，实物供电范围按所选型号手册确认。

定时电容、CONT 电容和电源去耦电容分别接不同节点。讲义第 77 页建议 CONT 用 `10 nF`，电子琴图上相关电容标值又有不同；读图时应识别每个电容的功能，不要只凭编号套用参数。


## 20. 一页复习表

| 知识点 | 关键结论 |
|---|---|
| 八个引脚 | 1 地、2 触发、3 输出、4 复位、5 控制、6 阈值、7 放电、8 电源 |
| 内部参考 | 默认约 1/3 VCC、2/3 VCC，由等值电阻分压形成 |
| 比较器 | 判断输入电压与参考电压的大小关系 |
| SR 状态记忆 | 置位、复位，其他时候保持原状态 |
| RESET | 低有效，强制输出低并导通放电开关 |
| 标准无稳态接线 | 2、6 脚并接电容节点；R_A 接 VCC—7，R_B 接 7—电容节点 |
| 充电阶段 | OUT 高，放电开关关断，经 R_A+R_B 充电 |
| 放电阶段 | OUT 低，放电开关导通，电容经 R_B 放电 |
| OUT 与电容波形 | OUT 为脉冲波形，电容为指数充放电波形 |
| 0.693 | ln(2) 的近似，来自阈值间变化；RC 才是时间常数 |
| 音高 | 基频增加，音高升高；按键改变等效定时电阻 |
| 实物边界 | 查供电、脚序、输出负载与去耦，区分 555 的具体类型 |

### 本篇核心公式

```text
t_H ≈ 0.693(R_A+R_B)C
t_L ≈ 0.693R_B C
T ≈ 0.693(R_A+2R_B)C
f = 1/T
D_H = (R_A+R_B)/(R_A+2R_B)

RC 时间常数：τ=RC
标准单稳态脉冲宽度：t≈1.1RC
```

记公式时同时记住接线和适用条件：改变控制电压、充放电路径或工作方式后，不能直接照搬同一组公式。


## 21. 自测题

1. NE555 的 2、3、4、6、7 脚分别是什么功能？
2. 5 V 供电、默认参考条件下，上下阈值约是多少？
3. 为什么电容处于两个阈值之间时，电路需要记住上一状态？
4. 标准振荡电路充电时，OUT 和 7 脚内部开关分别是什么状态？
5. 充电通过哪些电阻，放电主要通过哪个电阻？
6. R_A=1 kΩ、R_B=100 kΩ、C=100 nF 时，周期和频率约是多少？
7. 只把 C 增大到原来的 10 倍，理想频率和高电平占空比怎样变化？
8. `R_B/(R_A+2R_B)` 描述的是高电平占空比吗？
9. 普通 RC 从 0 V 向固定电压充电，达到最终电压约 63.2% 时经过了多少时间？NE555 中的 0.693 又是什么？
10. 电子琴中增大等效 R_B，音高怎样变化？为什么简单等差电阻链不保证准确音阶？
11. 接喇叭前串联一个电容，是否就能保证 NE555 不会过载？
12. 标准单稳态中 R=100 kΩ、C=10 μF，理想脉冲宽度约是多少？

### 参考答案

1. 2 是触发输入，3 是输出，4 是低有效复位，6 是阈值输入，7 是放电端。
2. 下阈值约 1.67 V，上阈值约 3.33 V。
3. 相同的中间电压可能处于充电或放电阶段，记忆决定继续哪个过程。
4. OUT 高，7 脚内部放电开关关断。
5. 经 R_A+R_B 充电；主要经 R_B 和 7 脚开关放电。
6. T 约 13.93 ms，f 约 71.8 Hz。
7. 频率降为约 1/10，高电平占空比在理想条件下不变。
8. 它是低电平时间比例，高电平占空比为 `(R_A+R_B)/(R_A+2R_B)`。
9. t=RC；0.693 约为 ln(2)，在标准振荡电路中来自特定阈值间的指数变化。
10. 周期增大、频率降低、音高降低；音阶遵循频率比例关系，等差电阻并不直接对应它。
11. 不能。电容隔离直流分量，输出电流、功耗、负载阻抗和驱动方式仍需匹配。
12. t≈1.1×100000×10×10⁻⁶≈1.1 s，前提是接线和触发满足标准条件。


## 22. 与原课程资料的对应关系

- 视频 `40_ne555简介.mp4`、`41_ne555引脚简介.mp4`：对应定时器用途、八个引脚与基本应用。
- 视频 `42_比较器（运放）.mp4`：对应比较器的输入关系和状态判断。
- 视频 `43_相反器.mp4`、`44_或非门.mp4`：对应反相器、或非门与真值关系。
- 视频 `45_双稳态触发器.mp4`：对应 SR 状态记忆和交叉反馈。
- 视频 `46_ne555的原理图绘制.mp4`：对应内部参考、比较器、状态逻辑、输出与放电功能的组合。
- 视频 `47_外围电路搭建.mp4`、`48_ne555输出方波原理.mp4`、`49_ne555外围电路.mp4`：对应标准 RC 接线、充放电过程、周期与占空比。
- 视频 `50_ne555电子琴原理介绍.mp4`：对应音高、频率与按键电阻链。
- 视频 `51_模拟器中的ne555.mp4`：对应使用模型和波形观察振荡行为。
- 配套 `ppt/模拟电路.pdf`：已在线核对第 **70—84 页**。其中 70—76 页介绍器件及内部功能，77—80 页介绍外围接线、仿真和原理，81—84 页介绍声音、电子琴电路与演奏示例。
- 原资料：[百度网盘课程分享](https://pan.baidu.com/s/1AZwoEpfx_2anpJ8p9ofD-Q?pwd=9988)，提取码 `9988`。
- 必要补充参考：[TI NE555 产品说明与数据手册入口](https://www.ti.com/product/NE555)，用于核对触发/阈值方向、复位优先级、放电关系与供电说明；实际器件以对应厂商和完整型号手册为准。

本篇依据课程目录和配套 PDF 在线预览整理，**视频未逐字转录**。公式推导、数值计算、单稳态说明、音高反算、仿真练习和硬件排查为学习补充；未实际执行的实验用“预期”或“待记录”标明。

涉及讲义容易造成误解的位置，本篇按电路关系澄清了 OUT 翻转方向、时间常数与 ln(2)、高低时间比例、供电和输出负载条件。保留课程学习顺序，实物实现仍需结合数据手册。

下一篇：**07-PCB 设计与 EDA 入门**，进入 PCB 概念、制造流程、EDA 软件、原理图绘制、PCB 布局与下单流程。
